# 🎭 Role & System Prompting

**Day 2 — Notebook 4 of 5 | Estimated Time: 20 minutes**

---

## What You'll Learn
- How to assign personas/roles to the model for better responses
- How to use system instructions in Gemini
- How roles affect tone, depth, and style of outputs

---

## What is Role Prompting?

**Role prompting** tells the model to adopt a specific identity, expertise, or persona. This shapes:
- **Tone** — formal, casual, encouraging
- **Depth** — expert-level detail vs. simplified explanations
- **Perspective** — technical, business, educational
- **Vocabulary** — domain-specific terminology

### System Instructions vs. In-Prompt Roles
Gemini supports **system instructions** — a dedicated field for defining the model's behavior that persists across the entire conversation.

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")

---

## 1. In-Prompt Role Assignment

The simplest approach — describe the role directly in the prompt.

In [ ]:
# Same question, different roles
question = "What is machine learning?"

roles = [
    "You are a kindergarten teacher. Explain in very simple terms a 5-year-old would understand.",
    "You are a senior ML engineer at Google. Give a technical explanation.",
    "You are a business executive. Explain the business value and ROI potential.",
]

for role in roles:
    prompt = f"{role}\n\n{question}"
    response = client.models.generate_content(model=MODEL_ID, contents=prompt)
    print(f"\n{'='*60}")
    print(f"Role: {role[:60]}...")
    print(f"{'='*60}")
    print(response.text[:300])
    print("...")

---

## 2. System Instructions (The Gemini Way)

Gemini's `system_instruction` parameter is the **preferred** way to set the model's role. It's separate from the user prompt and persists across multiple messages.

In [ ]:
# Using system_instruction for a code reviewer persona
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Review this code:\ndef add(a, b): return a+b",
    config=types.GenerateContentConfig(
        system_instruction="""You are a senior Python code reviewer. 
        For every code review:
        1. Rate the code quality (1-5 stars)
        2. List any issues found
        3. Suggest improvements
        4. Show the improved code
        Be constructive and professional."""
    ),
)

Markdown(response.text)

In [ ]:
# Pirate persona for fun!
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Tell me about the Python programming language.",
    config=types.GenerateContentConfig(
        system_instruction="""You are a friendly pirate who is also a programmer. 
        Explain everything using pirate metaphors and language. 
        Use 'arr', 'matey', 'treasure', and other pirate terms.
        Keep it fun but informative."""
    ),
)

Markdown(response.text)

---

## 3. Practical System Instructions

Here are some real-world system instruction patterns.

In [ ]:
# Customer support agent
support_instruction = """
You are a customer support agent for CloudTech Solutions.

Rules:
- Always be polite and empathetic
- If you don't know the answer, say "Let me escalate this to our specialist team"
- Never make up product features
- Always ask if there's anything else you can help with
- Keep responses concise (under 100 words)
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents="My cloud server has been down for 3 hours and I'm losing business!",
    config=types.GenerateContentConfig(system_instruction=support_instruction),
)

Markdown(response.text)

In [ ]:
# Technical documentation writer
doc_instruction = """
You are a technical documentation writer. 

When documenting code or APIs:
- Use clear, concise language
- Always include: Description, Parameters, Returns, Example Usage
- Use markdown formatting
- Add type hints in examples
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Document this function:
    def search_documents(query, top_k=5, filter_metadata=None):
        # searches vector DB and returns matching documents
        pass""",
    config=types.GenerateContentConfig(system_instruction=doc_instruction),
)

Markdown(response.text)

---

## 🏋️ Exercise 1: Create a Domain Expert

Create a system instruction for a **data analyst** who always responds with:
- A brief analysis
- Key metrics highlighted
- A recommendation

In [ ]:
# Exercise 1: Data analyst persona
# TODO: Write the system instruction

analyst_instruction = """
YOUR SYSTEM INSTRUCTION HERE
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Our e-commerce store data for last month:
    - Total visitors: 50,000
    - Purchases: 1,200
    - Average order value: $85
    - Cart abandonment rate: 72%
    - Return rate: 15%""",
    config=types.GenerateContentConfig(system_instruction=analyst_instruction),
)

Markdown(response.text)

---

## 🏋️ Exercise 2: Two Perspectives

Ask the same question to two different personas and compare the responses.

In [ ]:
# Exercise 2: Compare perspectives
question = "Should our company adopt AI-powered customer service chatbots?"

# TODO: Define two different personas
persona_1 = "YOUR FIRST PERSONA (e.g., CTO focused on innovation)"
persona_2 = "YOUR SECOND PERSONA (e.g., HR Manager focused on employees)"

for i, persona in enumerate([persona_1, persona_2], 1):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=question,
        config=types.GenerateContentConfig(system_instruction=persona),
    )
    print(f"\n{'='*60}")
    print(f"Persona {i}: {persona[:50]}...")
    print(f"{'='*60}")
    print(response.text[:400])
    print()

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Gemini System Instructions | Docs | [ai.google.dev/gemini-api/docs/system-instructions](https://ai.google.dev/gemini-api/docs/system-instructions) |
| Role Prompting Guide | Blog | [promptingguide.ai/techniques/role-prompting](https://www.promptingguide.ai/techniques/) |

---

**Next up:** [05_Prompt_Design_Patterns.ipynb](./05_Prompt_Design_Patterns.ipynb) — Production-ready prompt templates